# Mini Project Checkpoint

In [ ]:
from pyspark.sql import SparkSession, functions as F, Window
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DoubleType, DateType

spark = SparkSession.builder.appName("module-03-dataframes").master("local[*]").getOrCreate()
spark.conf.set("spark.sql.shuffle.partitions", "4")

In [ ]:
municipalities_schema = StructType([
    StructField("municipality_id", IntegerType(), True),
    StructField("municipality_name", StringType(), True),
    StructField("canton", StringType(), True),
    StructField("population", IntegerType(), True),
])
accessibility_schema = StructType([
    StructField("municipality_id", IntegerType(), True),
    StructField("accessibility_score", DoubleType(), True),
])
poi_schema = StructType([
    StructField("municipality_id", IntegerType(), True),
    StructField("poi_count", IntegerType(), True),
])
property_schema = StructType([
    StructField("municipality_id", IntegerType(), True),
    StructField("property_value_index", DoubleType(), True),
])
transactions_schema = StructType([
    StructField("transaction_id", IntegerType(), True),
    StructField("municipality_id", IntegerType(), True),
    StructField("property_id", IntegerType(), True),
    StructField("sale_price", IntegerType(), True),
    StructField("sale_date", StringType(), True),
    StructField("property_type", StringType(), True),
])
pop_history_schema = StructType([
    StructField("municipality_id", IntegerType(), True),
    StructField("year", IntegerType(), True),
    StructField("population", IntegerType(), True),
])

municipalities = spark.read.option("header", True).schema(municipalities_schema).csv("datasets/module_02/municipalities.csv")
accessibility_scores = spark.read.option("header", True).schema(accessibility_schema).csv("datasets/module_02/accessibility_scores.csv")
poi_counts = spark.read.option("header", True).schema(poi_schema).csv("datasets/module_02/poi_counts.csv")
property_values = spark.read.option("header", True).schema(property_schema).csv("datasets/module_02/property_values.csv")
transactions = spark.read.option("header", True).schema(transactions_schema).csv("datasets/module_03/transactions.csv").withColumn("sale_date", F.to_date("sale_date"))
population_history = spark.read.option("header", True).schema(pop_history_schema).csv("datasets/module_03/population_history.csv")

for name, df in {
    "municipalities": municipalities,
    "accessibility_scores": accessibility_scores,
    "poi_counts": poi_counts,
    "property_values": property_values,
    "transactions": transactions,
    "population_history": population_history,
}.items():
    df.createOrReplaceTempView(name)

In [ ]:
latest_population = population_history.filter(F.col("year") == 2025).select(
    "municipality_id", F.col("population").alias("population_2025")
)
base = transactions.join(municipalities.select("municipality_id", "municipality_name", "canton"), "municipality_id")     .join(accessibility_scores, "municipality_id", "left")     .join(poi_counts, "municipality_id", "left")     .join(property_values, "municipality_id", "left")     .join(latest_population, "municipality_id", "left")     .withColumn("sale_year", F.year("sale_date"))     .withColumn("sale_price_millions", F.round(F.col("sale_price") / F.lit(1000000.0), 3))

report = base.groupBy("canton", "municipality_id", "municipality_name").agg(
    F.count("*").alias("transactions"),
    F.avg("sale_price").alias("avg_sale_price"),
    F.avg("accessibility_score").alias("avg_accessibility_score"),
    F.first("poi_count", ignorenulls=True).alias("poi_count"),
    F.first("property_value_index", ignorenulls=True).alias("property_value_index"),
    F.first("population_2025", ignorenulls=True).alias("population_2025"),
)
score = report.withColumn(
    "performance_score",
    F.round((F.col("avg_sale_price") / F.lit(1000000.0)) + F.col("avg_accessibility_score") + (F.col("poi_count") / F.lit(100.0)) + F.col("property_value_index"), 3)
)
w = Window.partitionBy("canton").orderBy(F.desc("performance_score"), F.asc("municipality_id"))
final_report = score.withColumn("rank_in_canton", F.row_number().over(w)).filter(F.col("rank_in_canton") <= 3).orderBy("canton", "rank_in_canton")
final_report.show(50)
final_report.explain("formatted")